# Test IRIS Telescope Integration (Mixed Protocols)

Ten notatnik służy do interaktywnego testowania nowej implementacji `TreeIrisObservatory` oraz konektorów `Pilar` i `IrisCCD`.

**Co robi ten notatnik?**
1. Uruchamia w tle lokalne serwery Mock (TCP dla Pilar, UDP dla IrisCCD).
2. Podmienia konfigurację, aby łączyć się z `localhost` zamiast prawdziwym sprzętem.
3. Inicjalizuje drzewo urządzeń IRIS.
4. Pozwala wykonywać komendy `get` i `put` bezpośrednio z poziomu komórek.

In [1]:
import asyncio
import logging
import sys
import os
from unittest.mock import MagicMock

# Dodajemy ścieżkę do głównego katalogu projektu, aby widzieć moduły obsrv
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../')))

from obsrv.tree_components.specialized_components.tree_iris import TreeIrisObservatory
from obsrv.ob_config import SingletonConfig

# Konfiguracja logowania (aby widzieć co się dzieje w konektorach)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger("JupyterIris")

## 1. Definicja Mocków Sprzętowych
Poniższe klasy symulują działanie fizycznego sprzętu (Teleskopu Pilar i Kamery UDP).

In [2]:
class MockPilarServer:
    """Prosty serwer TCP udający teleskop Pilar."""
    def __init__(self, port=5950):
        self.server = None
        self.host = '127.0.0.1'
        self.port = port
        self.responses = {
            "GET OBJECT.EQUATORIAL.RA": ("OBJECT.EQUATORIAL.RA=10.5", True),
            "GET OBJECT.EQUATORIAL.DEC": ("OBJECT.EQUATORIAL.DEC=-20.0", True),
            "SET POINTING.SETUP.FOCUS.POSITION=2000": ("POINTING.SETUP.FOCUS.POSITION=2000", True),
        }

    async def handle_client(self, reader, writer):
        try:
            while True:
                data = await reader.readline()
                if not data: break
                line = data.decode().strip()
                if not line: continue
                
                parts = line.split(' ', 1)
                if len(parts) < 2: continue
                cmd_id, command = parts[0], parts[1]
                
                # Logika odpowiedzi
                if command in self.responses:
                    val, ok = self.responses[command]
                    writer.write(f"{cmd_id} {val}\n".encode())
                    status = "COMMAND COMPLETE" if ok else "COMMAND FAILED"
                    writer.write(f"{cmd_id} {status}\n".encode())
                else:
                    writer.write(f"{cmd_id} COMMAND COMPLETE\n".encode())
                await writer.drain()
        except Exception:
            pass

    async def start(self):
        self.server = await asyncio.start_server(self.handle_client, self.host, self.port)
        print(f"[MockPilar] Listening on TCP {self.host}:{self.port}")
        asyncio.create_task(self.server.serve_forever())

class MockIrisCcdProtocol(asyncio.DatagramProtocol):
    def __init__(self):
        self.transport = None

    def connection_made(self, transport):
        # Tutaj protokół otrzymuje transport automatycznie od asyncio
        self.transport = transport

    def datagram_received(self, data, addr):
        msg = data.decode().strip()
        resp = "**** OKAY 0" if msg == "sync" else "**** OKAY 1"
        if "temp" in msg: resp = "**** OKAY -15.5"
        
        if self.transport:
            self.transport.sendto(resp.encode(), addr)
        
class MockIrisCcdServer:
    """Prosty serwer UDP udający kamerę."""
    def __init__(self):
        self.transport = None
        self.protocol = None

    async def start(self, port=8888):
        loop = asyncio.get_running_loop()
        # create_datagram_endpoint zwraca (transport, protocol)
        self.transport, self.protocol = await loop.create_datagram_endpoint(
            lambda: MockIrisCcdProtocol(), local_addr=('127.0.0.1', port)
        )
        print(f"[MockIris] Listening on UDP 127.0.0.1:{port}")

## 2. Uruchomienie Mocków i Konfiguracja
Uruchamiamy serwery w tle i podmieniamy `SingletonConfig`, aby IRIS wskazywał na `localhost`.

In [3]:
import confuse
from unittest.mock import MagicMock
import asyncio
import logging

# --- PANCERNA KLASA MockCfg ---
class MockCfg:
    """
    Udaje zachowanie biblioteki confuse.
    Nie rzuca błędów przy dostępie [], dopiero przy .get().
    """
    def __init__(self, value, exists=True, path="root"):
        self.value = value
        self._exists = exists
        self._path = path

    def __getitem__(self, key):
        # Confuse zwraca obiekt View nawet jeśli klucz nie istnieje
        new_path = f"{self._path}.{key}"
        if not self._exists:
            return MockCfg(None, exists=False, path=new_path)
        
        if isinstance(self.value, dict):
            if key not in self.value:
                # Klucza nie ma, ale zwracamy obiekt (oznaczony jako nieistniejący)
                return MockCfg(None, exists=False, path=new_path)
            return MockCfg(self.value[key], path=new_path)
        
        # Próba wejścia głębiej w wartość, która nie jest słownikiem
        return MockCfg(None, exists=False, path=new_path)

    def get(self, template=None):
        if not self._exists:
            # Tu rzucamy specyficzny błąd confuse, który kod może łapać
            raise confuse.exceptions.NotFoundError(f"Mock config not found: {self._path}")
        return self.value

    # Metody do iteracji (np. for component in config['components'])
    def keys(self):
        if self._exists and isinstance(self.value, dict):
            return self.value.keys()
        return []

    def __iter__(self):
        if self._exists and isinstance(self.value, dict):
            return iter(self.value)
        return iter([])

    def __contains__(self, item):
        if self._exists and isinstance(self.value, dict):
            return item in self.value
        return False
        
    def as_str_seq(self):
        # Emulacja pobierania listy stringów (np. flags)
        val = self.get()
        if isinstance(val, list):
            return [str(v) for v in val]
        return []

# --- Start serwerów (zabezpieczenie przed błędem zajętego portu) ---
try:
    pilar_mock = MockPilarServer()
    iris_mock = MockIrisCcdServer()
    await pilar_mock.start()
    await iris_mock.start()
except OSError:
    print("Serwery prawdopodobnie już działają (porty zajęte). Kontynuuję.")

# --- Pełna Konfiguracja Testowa ---
# Uzupełniona o pola, których może szukać Observatory (flags, coords itp.)
test_config_raw = {
    'tree': {
        'iris-notebook': {
            'timeout_multiplier': 0.8
        },
        'iris': {
            'observatory': {
                'comment': 'Test IRIS Telescope',
                'flags': ['production'],
                'lon': -70.2,
                'lat': -24.6,
                'elev': 2800,
                'epoch': 2000,
                'style': {'color': '#FF0000'},
                'protocol': 'ignored', 
                'components': {
                    'mount': {
                        'kind': 'telescope', 'device_number': 0, 
                        'protocol': 'pilar', 'address': '127.0.0.1:5950', 
                        'type': 'az', 'slew_timeout': 5
                    },
                    'focuser': {
                        'kind': 'focuser', 'device_number': 0, 
                        'protocol': 'pilar', 'address': '127.0.0.1:5950'
                    },
                    'camera': {
                        'kind': 'camera', 'device_number': 0, 
                        'protocol': 'iris_ccd', 'address': '127.0.0.1:8888'
                    },
                    'dome': {
                        'kind': 'dome', 'device_number': 0, 
                        'protocol': 'alpaca', 'address': 'http://127.0.0.1:11111/api/v1',
                        'slew_timeout': 10
                    }
                }
            }
        }
    }
}

# Owijamy w MockCfg
mock_conf_obj = MockCfg(test_config_raw)

# Patchujemy SingletonConfig
SingletonConfig.get_config = MagicMock(return_value=mock_conf_obj)
print("Konfiguracja zaktualizowana. MockCfg gotowy na wszystko.")

[MockPilar] Listening on TCP 127.0.0.1:5950
[MockIris] Listening on UDP 127.0.0.1:8888
Konfiguracja zaktualizowana. MockCfg gotowy na wszystko.


## 3. Inicjalizacja Drzewa IRIS
Tworzymy obiekt `TreeIrisObservatory`. W tym momencie konektory są przygotowywane (lazy loading).

In [4]:
tree_iris = TreeIrisObservatory('iris-notebook', observatory_name='iris')
print("Drzewo IRIS zainicjalizowane. Dostępne komponenty:")
for child in tree_iris._observatory.children:
    print(f" - {child}")

2026-01-21 13:22:17,712 - tree_iris - WARNING - Could not load configuration for iris: Unknown protocol: ignored. Available: ['alpaca', 'pilar', 'beso', 'iris_ccd', 'dummy']


Drzewo IRIS zainicjalizowane. Dostępne komponenty:


## 4. Testy Protokołu Pilar (Teleskop)
Pobieramy `rightascension` oraz testujemy równoległość zapytań.

In [5]:
# Proste pobranie (używamy children['mount'] zamiast .mount)
# W zależności od implementacji Observatory, dostęp może być przez children['mount']
mount = tree_iris._observatory.children['mount']
ra = await mount.get('rightascension')
print(f"Right Ascension: {ra} stopni (Mock zwraca 10.5h)")

KeyError: 'mount'

In [ ]:
# Test równoległości
mount = tree_iris._observatory.children['mount']
tasks = [
    mount.get('rightascension'),
    mount.get('declination'),
    mount.get('rightascension'),
    mount.get('declination'),
    mount.get('rightascension')
]
results = await asyncio.gather(*tasks)
print(f"Wyniki zapytań równoległych: {results}")

# Test PUT (ruch focusera)
focuser = tree_iris._observatory.children['focuser']
resp = await focuser.put('move', Position=2000)
print(f"Odpowiedź Focusera: {resp}")

In [ ]:
# Test PUT (ruch focusera)
resp = await tree_iris._observatory.focuser.put('move', Position=2000)
print(f"Odpowiedź Focusera: {resp}")

## 5. Testy Protokołu IrisCCD (Kamera)
Testujemy komunikację po UDP.

In [ ]:
# Pobranie statusu kamery
camera = tree_iris._observatory.children['camera']
state = await camera.get('camerastate')
print(f"Camera State (UDP): {state}")

# Pobranie temperatury (symulowane)
temp = await camera.get('ccdtemperature')
print(f"Camera Temp: {temp}")

## 6. Czyszczenie
Zatrzymanie serwerów po testach.

In [ ]:
pilar_mock.server.close()
await pilar_mock.server.wait_closed()
iris_mock.transport.close()
print("Mocki zatrzymane.")